# Deep Learning with PyTorch · Autograd: the computational graph and backpropagation

Macademia · For adults

> **Running in Colab:** Open this file at [colab.research.google.com](https://colab.research.google.com) with **File → Upload notebook**. For a GPU choose **Runtime → Change runtime type → T4 GPU**. Run the cells from top to bottom (Shift + Enter). PyTorch is preinstalled in Colab; to run on your own computer use `pip install torch torchvision`.

In [ ]:
import torch, random, numpy as np
torch.manual_seed(0); random.seed(0); np.random.seed(0)          # reproducibility
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('PyTorch', torch.__version__, '| device:', device)

## 1. One neuron
$z = w_1x_1 + w_2x_2 + b$, $y = \tanh(z)$, $L = (y - t)^2$

In [ ]:
x1, x2, t = torch.tensor(2.0), torch.tensor(0.5), torch.tensor(0.8)
w1 = torch.tensor(-0.5, requires_grad=True)
w2 = torch.tensor(1.2, requires_grad=True)
b = torch.tensor(0.3, requires_grad=True)
y = torch.tanh(w1 * x1 + w2 * x2 + b)
L = (y - t) ** 2
print('y =', y.item(), 'L =', L.item())
print(L.grad_fn, '->', L.grad_fn.next_functions)                     # graph nodes

## 2. Backpropagation and a manual check

In [ ]:
L.backward()
manual = 2 * (y - t) * (1 - y ** 2) * x1                              # chain rule
print('autograd:', w1.grad.item(), '| manual:', manual.item())

## 3. Numerical derivative
Central difference: $(L(\theta+h) - L(\theta-h)) / 2h$

In [ ]:
def loss(w):
    return ((torch.tanh(w * x1 + w2 * x2 + b) - t) ** 2).item()
h = 1e-4
print('numeric:', (loss(w1.item() + h) - loss(w1.item() - h)) / (2 * h))
print(torch.autograd.gradcheck(lambda w: torch.tanh(w * 2.0) ** 2, torch.tensor([0.3], dtype=torch.float64, requires_grad=True)))

## 4. Gradients accumulate

In [ ]:
w = torch.tensor(1.0, requires_grad=True)
for i in range(3):
    (w * 3).backward()
    print(i, w.grad.item())                                          # 3, 6, 9: accumulates
w.grad = None

## 5. Gradient descent by hand

In [ ]:
for step in range(20):
    y = torch.tanh(w1 * x1 + w2 * x2 + b)
    L = (y - t) ** 2
    L.backward()
    with torch.no_grad():                                            # keep the update out of the graph
        for p in (w1, w2, b):
            p -= 0.1 * p.grad
            p.grad = None
    if step % 5 == 0:
        print(step, round(L.item(), 5))

## Exercises
1. Set $x_1 = 0$: why is $w_1$'s gradient zero?
2. Cut $y$ off the graph with `detach()` and try `backward`: what happened?
3. Write a `gradcheck` for your own function.